<a href="https://colab.research.google.com/github/Adarsha2004/finetuning/blob/main/gemma_cft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q unsloth
!pip install -q datasets

In [2]:
import torch
import re
import random
import math

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.10.0+cu128
CUDA available: True


In [3]:
from unsloth import FastLanguageModel

BASE_MODEL = "unsloth/gemma-4-E2B-it"
MAX_SEQ_LENGTH = 1024
SEED = 42

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

print(f"\n[OK] Model loaded: {BASE_MODEL}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]


[OK] Model loaded: unsloth/gemma-4-E2B-it
Parameters: 4,358,766,112


In [35]:
def compute_perplexity(model, tokenizer, texts, max_length=512):
    """
    Calculates perplexity using a neutral assistant role.
    This avoids forcing the model into a 'summary' task and measures
    raw domain likelihood more accurately.
    """
    total_loss = 0
    total_tokens = 0

    model.eval()
    for text in texts:
        # Use a neutral prompt to establish domain context without a task
        messages = [
            {"role": "user", "content": [{"type": "text", "text": "Provide medical textbook information:"}]},
            {"role": "assistant", "content": [{"type": "text", "text": text}]}
        ]

        full_input_ids = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=False,
            return_tensors="pt"
        ).to(model.device)

        prompt_input_ids = tokenizer.apply_chat_template(
            messages[:1],
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to(model.device)

        prompt_len = prompt_input_ids.shape[1]

        labels = full_input_ids.clone()
        labels[:, :prompt_len] = -100

        with torch.no_grad():
            outputs = model(input_ids=full_input_ids, labels=labels)

        num_tokens = (labels != -100).sum().item()
        if num_tokens > 0:
            total_loss += outputs.loss.item() * num_tokens
            total_tokens += num_tokens

    if total_tokens == 0: return float('inf')

    avg_loss = total_loss / total_tokens
    return math.exp(avg_loss)

In [37]:
# Re-evaluating general text with neutral prompt
general_texts = [
    "The weather was beautiful that morning as the children walked to school through the park.",
    "Scientists have discovered a new species of butterfly in the Amazon rainforest.",
    "The recipe calls for two cups of flour, one egg, and a tablespoon of butter.",
    "The basketball game went into overtime after a dramatic three-point shot.",
    "She opened the book and began reading the first chapter aloud to her students.",
]

ppl = compute_perplexity(model, tokenizer, general_texts)
print(f"General Perplexity (Neutral Prompt): {ppl:.2f}")

General Perplexity (Neutral Prompt): 232.13


In [40]:
sample_medical = [
    "The patient presented with acute chest pain radiating to the left arm, accompanied by diaphoresis and shortness of breath, suggestive of myocardial infarction.",
    "Hemoglobin levels were recorded at 8.2 g/dL, indicating moderate anemia, and iron supplementation was initiated.",
    "MRI findings revealed a lesion in the temporal lobe consistent with early-stage glioma.",
    "The individual reported persistent hyperglycemia with fasting blood glucose levels above 140 mg/dL, indicating uncontrolled diabetes mellitus.",
    "Antibiotic therapy with amoxicillin-clavulanate was prescribed for bacterial sinusitis following clinical evaluation.",
]

medical_ppl = compute_perplexity(model, tokenizer, sample_medical)

print(f"Medical Perplexity (Neutral Prompt): {medical_ppl:.1f}")
print("(This is our more accurate baseline for CPT)")

Medical Perplexity (Neutral Prompt): 139.7
(This is our more accurate baseline for CPT)


In [42]:
from datasets import load_dataset

configs = [
    "Pathology_Robbins",
    "Physiology_Levy",
    "Pharmacology_Katzung",
    "Neurology_Adams",
    "Pediatrics_Nelson"
]

NUM_TRAIN = 5000
NUM_VAL = 500
TARGET_TOTAL = NUM_TRAIN + NUM_VAL

all_texts = []

print(f"[...] Collecting {TARGET_TOTAL} samples...")

# -----------------------------
# LOAD DATA
# -----------------------------
for cfg in configs:
    print(f"\n[+] Loading: {cfg}")

    ds = load_dataset(
        "zxvix/MedicalTextbook",
        cfg,
        split="train",
        streaming=True
    )

    for example in ds:
        text = example.get("text", "")

        if text and len(text.split()) > 100:
            all_texts.append(text)

        if len(all_texts) >= TARGET_TOTAL:
            break

    if len(all_texts) >= TARGET_TOTAL:
        break

print(f"\n[OK] Total collected: {len(all_texts)}")

[...] Collecting 5500 samples...

[+] Loading: Pathology_Robbins


README.md: 0.00B [00:00, ?B/s]


[+] Loading: Physiology_Levy

[+] Loading: Pharmacology_Katzung

[+] Loading: Neurology_Adams

[OK] Total collected: 5500


In [43]:
sample = all_texts[0]
words = sample.split()
print(f"Sample filing length: {len(words)} words")
print(f"\nFirst 200 words:")
print(" ".join(words[:200]))
print("\n...")
print(f"\nLast 100 words:")
print(" ".join(words[-100:]))

Sample filing length: 498 words

First 200 words:
Plasma Membrane: Protection and Nutrient Acquisition Biosynthetic Machinery: Endoplasmic Reticulum and Golgi Apparatus Waste Disposal: Lysosomes and Proteasomes Modular Signaling Proteins, Hubs, and Components of the Extracellular Matrix Proliferation and the Cell Cycle Pathology literally translates to the study of suffering (Greek pathos = suffering, logos = study); as applied to modern medicine, it is the study of disease. Virchow was certainly correct in asserting that disease originates at the cellular level, but we now realize that cellular disturbances arise from alterations in molecules (genes, proteins, and others) that influence the survival and behavior of cells. Thus, the foundation of modern pathology is understanding the cellular and molecular abnormalities that give rise to diseases. It is helpful to consider these abnormalities in the context of normal cellular structure and function, which is the theme of this introduct

In [44]:
# -----------------------------
# CLEAN TEXT
# -----------------------------
def clean_text(text: str) -> str:
    if not text:
        return ""

    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"<.*?>", "", text)

    # Fix merged words
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", text)
    text = re.sub(r"([a-zA-Z])(\d)", r"\1 \2", text)
    text = re.sub(r"(\d)([a-zA-Z])", r"\1 \2", text)

    # Fix commas spacing
    text = re.sub(r",([a-zA-Z])", r", \1", text)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

cleaned_texts = [clean_text(t) for t in all_texts]


In [45]:
# -----------------------------
# SHUFFLE + SPLIT (IMPORTANT)
# -----------------------------
random.shuffle(cleaned_texts)

train_texts = cleaned_texts[:NUM_TRAIN]
val_texts   = cleaned_texts[NUM_TRAIN:NUM_TRAIN + NUM_VAL]

print(f"\nTrain filings: {len(train_texts)}")
print(f"Val filings:   {len(val_texts)}")


Train filings: 5000
Val filings:   500


In [46]:
# -----------------------------
# CHUNKING
# -----------------------------
def chunk_texts(texts, chunk_size=512, overlap=0.1):
    step = int(chunk_size * (1 - overlap))
    all_chunks = []

    for text in texts:
        words = text.split()
        for i in range(0, len(words), step):
            chunk = words[i:i + chunk_size]
            if len(chunk) > 50:
                all_chunks.append(" ".join(chunk))

    return all_chunks

train_chunks = chunk_texts(train_texts, chunk_size=512, overlap=0.1)
val_chunks   = chunk_texts(val_texts, chunk_size=512, overlap=0.1)

print(f"\nTrain chunks: {len(train_chunks)}")
print(f"Val chunks:   {len(val_chunks)}")


Train chunks: 7262
Val chunks:   721


In [47]:
# -----------------------------
# SAMPLE OUTPUT
# -----------------------------
sample = train_chunks[0].split()

print("\nSample chunk (first 100 words):")
print(" ".join(sample[:100]))

print("\n...")
print("\nLast 50 words:")
print(" ".join(sample[-50:]))


Sample chunk (first 100 words):
Polyostotic fibrous dysplasia may continue to cause problems into adulthood. If it involves the limb girdles, it can cause crippling deformities and fractures. The Mc Cune-Albright syndrome usually presents with precocious sexual development, most often in girls. The skeletal manifestations are managed as for other polyostotic fibrous dysplasia, whereas the endocrinopathies are treated medically. Metastatic tumors greatly outnumber primary bone cancers. The pathways of tumor spread to bone include (1) direct extension, (2) lymphatic or hematogenous dissemination, and (3) intraspinal seeding (via the Batson plexus of veins). Any cancer can spread to bone, but in adults more than 75% of

...

Last 50 words:
bone formation may produce sclerotic metastases. The presence of bone metastases carries a poor prognosis. Therapeutic options include systemic chemotherapy, radiation, and bisphosphonates. Surgery may be necessary to stabilize pathologic fractures. Fi

In [48]:
# Convert to HuggingFace Dataset format
from datasets import Dataset

train_dataset = Dataset.from_dict({"text": train_chunks})
val_dataset = Dataset.from_dict({"text": val_chunks})

print(f"Train dataset: {train_dataset}")
print(f"Val dataset:   {val_dataset}")

Train dataset: Dataset({
    features: ['text'],
    num_rows: 7262
})
Val dataset:   Dataset({
    features: ['text'],
    num_rows: 721
})


In [49]:
# IMPORTANT: Measure base model perplexity on the ACTUAL val chunks
# We need this before we reload the model for training
val_sample = val_chunks[:50]
base_ppl_val = compute_perplexity(model, tokenizer, val_sample)
print(f"Base model perplexity on medical validation chunks: {base_ppl_val:.1f}")
print("(We'll compare this to the CPT model later)")

Base model perplexity on medical validation chunks: 48.0
(We'll compare this to the CPT model later)


In [51]:
# Free memory from the inference model
del model
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

print(f"[OK] Fresh model loaded for training")

NameError: name 'model' is not defined